# SARIMAX + FFNN Mejorado

_Generado: 2025-10-18 03:24_

Notebook con bloques 'drop‑in' listos para ejecutar en orden.

## 1) Imports y configuración

In [ ]:
# === [BLOQUE 1] IMPORTS Y CONFIGURACIÓN ===
import numpy as np
import pandas as pd
import yfinance as yf
import tensorflow as tf
import matplotlib.pyplot as plt  # Para diagnósticos puntuales
import plotly.graph_objects as go

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import regularizers

# -------- Parámetros base (ajústalos si quieres) --------
TICKER = "AAPL"                    # Activo
RANDOM_SEED = 42                   # Reproducibilidad
N_LAG = 20                         # Ventana FFNN
EPOCHS = 150                       # Más estable con EarlyStopping
BATCH_SIZE = 32
PRED_STEPS = 5                     # 5 días
VALID_DAYS = 20                    # Ventana para MAPE de validación

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

def mape(y_true, y_pred):
    y_true = np.asarray(y_true).astype(float)
    y_pred = np.asarray(y_pred).astype(float)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100.0


## 2) Descarga de datos, limpieza y split (incluye fechas futuras)

In [ ]:
# === [BLOQUE 2] DESCARGA Y PREPARACIÓN DE DATOS ===
# Descargamos 5 años para dar contexto suficiente a SARIMAX y FFNN
hist = yf.Ticker(TICKER).history(period="5y")[["Close"]].dropna().copy()
hist.index = pd.to_datetime(hist.index)
hist = hist.sort_index()

# Split para validación: últimos VALID_DAYS se reservan para medir MAPE de cada modelo
train = hist.iloc[:-VALID_DAYS]
valid = hist.iloc[-VALID_DAYS:]

# Fechas objetivo de predicción (5 días hábiles hacia adelante desde el último día del histórico)
last_date = hist.index[-1]
# Genera exactamente 5 hábiles hacia delante
future_dates = pd.bdate_range(last_date + pd.Timedelta(days=1), periods=PRED_STEPS)

train.tail(), valid.head(), future_dates


## 3) SARIMAX con búsqueda pequeña y tolerante

In [ ]:
# === [BLOQUE 3] SARIMAX ===
import warnings
warnings.filterwarnings("ignore")

candidate_orders = [(p,d,q) for p in range(0,3) for d in (0,1) for q in range(0,3)]
best_aic = np.inf
best_order = None
best_model = None

y_train = train["Close"]

for order in candidate_orders:
    try:
        model = SARIMAX(y_train, order=order, enforce_stationarity=False, enforce_invertibility=False)
        res = model.fit(disp=False)
        if res.aic < best_aic:
            best_aic = res.aic
            best_order = order
            best_model = res
    except Exception:
        continue

if best_model is None:
    # Respaldo si todo falla
    best_order = (1,1,1)
    best_model = SARIMAX(y_train, order=best_order, enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

# Predicción en VALID_DAYS para MAPE (validación)
sarimax_valid_pred = best_model.get_forecast(steps=VALID_DAYS).predicted_mean
sarimax_valid_pred.index = valid.index  # alinear por fecha
mape_sarimax_valid = mape(valid["Close"], sarimax_valid_pred)

# Reentrenar con todo el historial y proyectar 5 días
sarimax_full = SARIMAX(hist["Close"], order=best_order, enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
sarimax_future = sarimax_full.get_forecast(steps=PRED_STEPS).predicted_mean
sarimax_future.index = future_dates

print(f"Mejor orden SARIMAX: {best_order} | AIC: {best_aic:.1f} | MAPE valid: {mape_sarimax_valid:.2f}%")
sarimax_future.head()


## 4) FFNN robusta (regularización + early stopping)

In [ ]:
# === [BLOQUE 4] FFNN ===
def make_supervised(series, n_lag):
    X, y = [], []
    vals = series.values.astype(float)
    for i in range(n_lag, len(vals)):
        X.append(vals[i-n_lag:i])
        y.append(vals[i])
    return np.array(X), np.array(y)

# Escalamos a [0,1] solo la columna Close
scaler = MinMaxScaler()
scaled = scaler.fit_transform(hist[["Close"]])

scaled_df = pd.DataFrame(scaled, index=hist.index, columns=["Close"])
scaled_train = scaled_df.iloc[:-VALID_DAYS]
scaled_valid = scaled_df.iloc[-VALID_DAYS:]

# Conjuntos supervisados
X_train, y_train_ff = make_supervised(scaled_train["Close"], N_LAG)
# Para valid/pred, necesitamos que haya suficientes rezagos
tail_for_valid = scaled_df["Close"].iloc[-(VALID_DAYS + N_LAG):]
X_valid, y_valid_ff = make_supervised(tail_for_valid, N_LAG)

# Modelo FFNN con regularización y BN
model = Sequential([
    Dense(128, activation="relu", input_shape=(N_LAG,), kernel_regularizer=regularizers.l2(1e-4)),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
    BatchNormalization(),
    Dropout(0.2),
    Dense(1, activation="linear")
])

model.compile(optimizer="adam", loss="mse")
callbacks = [
    EarlyStopping(patience=20, restore_best_weights=True, monitor="val_loss"),
    ReduceLROnPlateau(patience=10, factor=0.5, min_lr=1e-5)
]

history = model.fit(
    X_train, y_train_ff,
    validation_data=(X_valid, y_valid_ff),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=0,
    callbacks=callbacks
)

# MAPE en valid (invertimos escala)
y_valid_pred_scaled = model.predict(X_valid, verbose=0).flatten()
y_valid_pred = scaler.inverse_transform(y_valid_pred_scaled.reshape(-1,1)).flatten()
y_valid_real = scaler.inverse_transform(y_valid_ff.reshape(-1,1)).flatten()
mape_ffnn_valid = mape(y_valid_real, y_valid_pred)

print(f"MAPE valid FFNN: {mape_ffnn_valid:.2f}%")

# --- Pronóstico recursivo 5 pasos hacia delante ---
last_window = scaled_df['Close'].values[-N_LAG:].copy()
ffnn_future_scaled = []
for _ in range(PRED_STEPS):
    x = last_window.reshape(1, -1)
    yhat = model.predict(x, verbose=0).item()
    ffnn_future_scaled.append(yhat)
    last_window = np.roll(last_window, -1)
    last_window[-1] = yhat

ffnn_future = scaler.inverse_transform(np.array(ffnn_future_scaled).reshape(-1,1)).flatten()
ffnn_future = pd.Series(ffnn_future, index=future_dates, name='FFNN_pred')

ffnn_future.head()


## 5) Tabla final + gráfica comparativa

In [ ]:
# === [BLOQUE 5] TABLA FINAL Y GRÁFICA ===
final_df = pd.DataFrame({
    "pred_close_sarimax": sarimax_future,
    "pred_close_ffnn": ffnn_future
})

final_df.index.name = "date"

print("=== MAPE de validación ===")
print(f"SARIMAX: {mape_sarimax_valid:.2f}%")
print(f"FFNN   : {mape_ffnn_valid:.2f}%\n")

print("=== Predicciones 5 días hábiles ===")
print(final_df.round(2).to_string())

# Gráfica comparando histórico reciente vs pronósticos
lookback = 120  # días para mostrar contexto
hist_tail = hist.tail(lookback)

fig = go.Figure()
fig.add_trace(go.Scatter(x=hist_tail.index, y=hist_tail["Close"],
                         mode="lines", name="Histórico",
                         line=dict(width=2)))

fig.add_trace(go.Scatter(x=final_df.index, y=final_df["pred_close_sarimax"],
                         mode="lines+markers", name="SARIMAX (5d)",
                         line=dict(dash="dash", width=3)))

fig.add_trace(go.Scatter(x=final_df.index, y=final_df["pred_close_ffnn"],
                         mode="lines+markers", name="FFNN (5d)",
                         line=dict(dash="dot", width=3)))

fig.update_layout(
    title=f"Pronóstico de {TICKER}: 5 días hábiles (SARIMAX vs FFNN)",
    xaxis_title="Fecha",
    yaxis_title="Precio de Cierre (USD)",
    template="plotly_white",
    width=900, height=500
)
fig.show()
